# pyNCBIGene: NCBI Gene annotation in parquet

Python companion to the Bioconductor [RNCBIGene](https://github.com/vjcitn/RNCBIGene) package.
All queries run through a persistent [duckdb](https://duckdb.org) connection that reads
Apache Parquet files directly from an NSF Open Storage Network bucket, pushing filters
and column selections to the parquet layer before any data lands in Python.

The return type of `open_ncbi_gene()` is a `duckdb.DuckDBPyRelation` -- a lazy query object.
Chain `.filter()`, `.select()`, and `.df()` (collect to pandas) as needed.

In [ ]:
from pyNCBIGene import (
    available_ncbi_parquet,
    ncbi_gene_fields,
    ncbi_parquet_info,
    open_ncbi_gene,
    map_ids_ng,
    join_ncbi_gene,
)
import pandas as pd

## 1. Scope: available resources

Eight parquet files live in the OSN bucket, covering all organisms in NCBI Gene.

In [ ]:
available_ncbi_parquet()

Record counts -- `COUNT(*)` pushed to duckdb, reads row-group metadata without fetching data.

In [ ]:
resources = [r.replace('.parquet', '') for r in available_ncbi_parquet()]
counts = {r: open_ncbi_gene(r).count('*').fetchone()[0] for r in resources}
pd.Series(counts, name='n_rows').sort_index()

Bucket metadata -- sizes, upload timestamps, NCBI source dates.

In [ ]:
ncbi_parquet_info()

## 2. Lazy remote queries with `open_ncbi_gene()`

`open_ncbi_gene()` returns a lazy `DuckDBPyRelation` backed by a duckdb VIEW over the
remote parquet file.  No data is fetched until `.df()` is called.

In [ ]:
tbl = open_ncbi_gene('gene_info', taxid=9606)
type(tbl)

Compose a filter + select before collecting -- only matched rows and columns travel the wire.

In [ ]:
(
    open_ncbi_gene('gene_info', taxid=9606)
    .filter('"Symbol" IN (\'TP53\', \'ORMDL3\', \'BRCA1\', \'GSDMB\')')
    .select('"Symbol", "GeneID", "map_location", "type_of_gene"')
    .df()
)

When `taxid` is specified the `#tax_id` column is dropped from the result automatically.
Omitting `taxid` returns a relation spanning all organisms:

In [ ]:
# TP53 orthologs across all taxa (returns many rows)
open_ncbi_gene('gene_orthologs').filter('"GeneID" = 7157').df().head(10)

## 3. Discovering available fields with `ncbi_gene_fields()`

In [ ]:
ncbi_gene_fields('gene_info')

In [ ]:
ncbi_gene_fields('gene2go')

In [ ]:
# programmatic validation
gi_cols = ncbi_gene_fields('gene_info')['column_name'].tolist()
go_cols = ncbi_gene_fields('gene2go')['column_name'].tolist()
print('map_location in gene_info:', 'map_location' in gi_cols)
print('map_location in gene2go:  ', 'map_location' in go_cols)

## 4. Identifier mapping with `map_ids_ng()`

All filtering is pushed to duckdb before a single `.df()` fetch.  Returns a dict;
`None` for unrecognised keys.

### Symbol to GeneID

In [ ]:
map_ids_ng(['ORMDL3', 'TP53', 'GSDMB', 'XyZZY'], keytype='Symbol',
           column='GeneID', taxid=9606)

### Symbol to chromosomal map location

In [ ]:
map_ids_ng(['ORMDL3', 'TP53', 'GSDMB', 'BRCA1'], keytype='Symbol',
           column='map_location', taxid=9606)

### Ensembl to Symbol

A JOIN between `gene2ensembl` and `gene_info` runs entirely in duckdb.

In [ ]:
map_ids_ng(
    ['ENSG00000073605', 'ENSG00000141510', 'ENSG00000012048'],
    keytype='Ensembl', column='Symbol', taxid=9606
)

## 5. Non-human organisms

All resources cover all NCBI organisms.  Switching species is a matter of changing `taxid`.
Here we look up Ensembl gene and transcript identifiers for the mouse (_Mus musculus_,
taxid 10090) gene _Lilrb4a_.

In [ ]:
# quick Symbol -> Ensembl gene ID
map_ids_ng(['Lilrb4a'], keytype='Symbol', column='Ensembl', taxid=10090)

For the full transcript-level picture, join `gene_info` and `gene2ensembl` in duckdb.
Both relations share the same connection so the join executes before any data is collected.

In [ ]:
from pyNCBIGene._state import get_connection

info = (
    open_ncbi_gene('gene_info', taxid=10090)
    .filter('"Symbol" = \'Lilrb4a\'')
    .select('"GeneID", "Symbol"')
)
g2e = (
    open_ncbi_gene('gene2ensembl', taxid=10090)
    .select('"GeneID", "Ensembl_gene_identifier", "Ensembl_rna_identifier", "Ensembl_protein_identifier"')
)

con = get_connection()
con.register('_info', info.df())
con.register('_g2e',  g2e.df())
con.sql(
    'SELECT i."Symbol", g."Ensembl_gene_identifier", '
    'g."Ensembl_rna_identifier", g."Ensembl_protein_identifier" '
    'FROM _info i JOIN _g2e g USING ("GeneID")'
).df()

The result reveals two distinct Ensembl gene models for _Lilrb4a_ -- something
a two-step approach (Symbol -> one gene ID -> transcripts) would miss.

## 6. Joining local data to remote annotation with `join_ncbi_gene()`

`join_ncbi_gene()` validates the join columns against both the local DataFrame and the
remote resource, then executes the join in duckdb.

In [ ]:
local_df = pd.DataFrame({'Symbol': ['ORMDL3', 'TP53', 'BRCA1', 'GSDMB', 'XyZZY']})

join_ncbi_gene(local_df, by='Symbol', resource='gene_info', taxid=9606) \
    .select('"Symbol", "GeneID", "map_location", "type_of_gene"') \
    .df()

Unmatched keys (`XyZZY`) appear with `NaN` in annotation columns because the default
join type is `'left'`.  Use `how='inner'` to drop them.

### GO terms and RefSeq: separate joins required

GO terms and RefSeq IDs are both one-to-many from GeneID.  Joining both resources in a
single query would produce a Cartesian product -- the correct pattern is two independent joins.

In [ ]:
local_ids = pd.DataFrame({'GeneID': [94103, 7157, 672]})

# GO annotations
go_df = (
    join_ncbi_gene(local_ids, by='GeneID', resource='gene2go',
                  taxid=9606, how='inner')
    .select('"GeneID", "GO_ID", "GO_term", "Evidence", "Category"')
    .df()
)
print(f'GO rows: {len(go_df)}')
go_df.head()

In [ ]:
# RefSeq identifiers
refseq_df = (
    join_ncbi_gene(local_ids, by='GeneID', resource='gene2refseq',
                  taxid=9606, how='inner')
    .select('"GeneID", "RNA_nucleotide_accession.version", "protein_accession.version"')
    .filter('"RNA_nucleotide_accession.version" != \'-\'')
    .df()
)
print(f'RefSeq rows: {len(refseq_df)}')
refseq_df.head()

## 7. Local caching with `cache_by_taxon()`

All queries above hit the OSN bucket on every call.  `cache_by_taxon()` performs a
one-time filter of each remote parquet to a single taxon and stores the result locally.
After that, `open_ncbi_gene()` routes to the local file automatically.

Requires `pybiocfilecache`: `pip install pybiocfilecache`

In [ ]:
# from pyNCBIGene import cache_by_taxon, taxon_cache_info, clear_taxon_cache

# one-time setup -- filters and writes local parquet for each resource
# cache_by_taxon(9606)        # human: all eight resources (slow)
# cache_by_taxon(9606, resources=['gene_info', 'gene2go'])  # subset

# check what is cached
# taxon_cache_info(9606)

# identical call to open_ncbi_gene -- routed to local file automatically
# open_ncbi_gene('gene_info', taxid=9606).filter(...).df()

# remove live cache and route back to OSN
# clear_taxon_cache(9606)

print('Caching cells are commented out to avoid network/disk costs in this demo.')
print('Uncomment and run interactively after pip install pybiocfilecache.')

## 8. Reproducibility snapshots with `freeze_taxon_cache()`

A live cache can be overwritten at any time.  `freeze_taxon_cache()` makes a physical
copy under an identifying tag.  Frozen entries survive `clear_taxon_cache()`.
Duplicate tags are rejected unless `force=True`.

In [ ]:
# from pyNCBIGene import freeze_taxon_cache

# freeze_taxon_cache(9606, tag='paper_2026_07')  # snapshot live cache

# query the frozen snapshot via freeze_tag=
# open_ncbi_gene('gene_info', taxid=9606, freeze_tag='paper_2026_07') \
#     .filter('"Symbol" = \'TP53\'').df()

# stop with KeyError if tag not found -- fails loudly, not silently
# open_ncbi_gene('gene_info', taxid=9606, freeze_tag='nonexistent')

print('Freeze cells are commented out -- requires prior cache_by_taxon() call.')

## Session info

In [ ]:
import sys, duckdb, pandas
import pyNCBIGene

print(f'Python:      {sys.version}')
print(f'pyNCBIGene:  {pyNCBIGene.__version__}')
print(f'duckdb:      {duckdb.__version__}')
print(f'pandas:      {pandas.__version__}')